In [2]:
import pandas as pd
import matplotlib.pyplot as plt

raw_data = pd.read_csv("../data/analysis_table/data_cleaned.csv")
session_df = pd.read_csv("../data/analysis_table/session_table.csv")
user_df = pd.read_csv("../data/analysis_table/user_table.csv")
funnel = pd.read_csv("../data/analysis_table/funnel_table.csv")

In [3]:
raw_data['product_category'] = raw_data['category_code'].str.split(".").str[0]

# Goal: understanding which product category has a higher conversion rate (from view to purchase)
# we don't care about cart for now

view_purchase_df = raw_data[raw_data['event_type'].isin(['view','purchase'])]

In [4]:
product_conversion = (view_purchase_df.groupby(['product_id', 'product_category']).agg(
    has_view = ('event_type', lambda x: (x == 'view').any()),
    has_purchase = ('event_type', lambda x: (x == 'purchase').any())
    ).reset_index()
)

product_conversion.head()

,product_id,product_category,has_view,has_purchase
0,1000978,electronics,True,True
1,1001588,electronics,True,True
2,1001606,electronics,True,False
3,1001618,electronics,True,True
4,1001619,electronics,True,True


In [5]:
category_conversion = (product_conversion.groupby('product_category').agg(
    products_viewed = ('has_view', 'sum'),
    products_purchased = ('has_purchase', 'sum')
    ).reset_index()
    )

category_conversion['conversion_rate'] = (category_conversion['products_purchased'] / category_conversion['products_viewed']).round(2)

category_conversion = category_conversion.sort_values('conversion_rate', ascending = False)

category_conversion

,product_category,products_viewed,products_purchased,conversion_rate
10,medicine,27,12,0.44
3,auto,1400,379,0.27
2,appliances,11927,3136,0.26
7,electronics,14300,2847,0.20
4,computers,7758,1508,0.19
8,furniture,6873,884,0.13
0,accessories,2538,265,0.10
1,apparel,13562,1416,0.10
5,construction,5151,486,0.09
9,kids,4757,427,0.09


In [9]:
# AOV (Average Order Value) by category

purchase_df = raw_data[raw_data['event_type'] == 'purchase']

order_value = (purchase_df.groupby(['user_session', 'product_category'])['price'].sum().reset_index(name = 'order_value'))

aov_by_category = (order_value.groupby('product_category')['order_value'].mean().sort_values(ascending = False).reset_index(name = 'AOV')).round(2)

aov_by_category

,product_category,AOV
0,electronics,492.59
1,computers,443.06
2,sport,262.15
3,furniture,256.47
4,appliances,209.29
5,country_yard,153.36
6,auto,133.93
7,construction,132.14
8,kids,127.38
9,apparel,95.13


In [11]:
product_summary_by_category = pd.merge(aov_by_category, category_conversion, on = 'product_category', how = 'left')
product_summary_by_category

,product_category,AOV,products_viewed,products_purchased,conversion_rate
0,electronics,492.59,14300,2847,0.20
1,computers,443.06,7758,1508,0.19
2,sport,262.15,2084,139,0.07
3,furniture,256.47,6873,884,0.13
4,appliances,209.29,11927,3136,0.26
5,country_yard,153.36,147,5,0.03
6,auto,133.93,1400,379,0.27
7,construction,132.14,5151,486,0.09
8,kids,127.38,4757,427,0.09
9,apparel,95.13,13562,1416,0.10
